# 🔴 Delhivery Logistics Network — Bottleneck Analysis
### Notebook 4 of 7

**Objective:** Identify which hubs are structural bottlenecks  
and which corridors are the biggest contributors to SLA breaches.

**Metrics we compute:**
- Betweenness Centrality — which hubs sit on the most paths
- In-degree and Out-degree — how many routes flow through each hub
- Clustering Coefficient — how interconnected a hub's neighbors are
- Bottleneck Score — combined metric ranking hubs by risk

**Output:** Top 5 bottleneck hubs for the Strategy Memo

---

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('../outputs/visualisations', exist_ok=True)
os.makedirs('../outputs/model_results', exist_ok=True)

print("✅ Libraries loaded")

## 📂 Step 1 — Load Graph and Clean Data

In [ ]:
# Load clean dataset
df = pd.read_csv('../data/delivery_data_clean.csv')
df['od_start_time'] = pd.to_datetime(df['od_start_time'])

# Load saved graph
G = nx.read_graphml('../outputs/model_results/logistics_graph.graphml')

print("=" * 55)
print("       GRAPH AND DATA LOADED")
print("=" * 55)
print(f"\n  Dataset rows     : {len(df):,}")
print(f"  Graph nodes      : {G.number_of_nodes():,}")
print(f"  Graph edges      : {G.number_of_edges():,}")
print(f"  Graph directed   : {G.is_directed()}")
print("\n" + "=" * 55)

## 📐 Step 2 — Compute Betweenness Centrality

Betweenness centrality measures how often a node appears  
on the shortest path between other nodes.

A hub with high betweenness centrality is a **structural chokepoint** —  
if it fails or gets congested, it disrupts the maximum number  
of other routes in the network.

This is the single most important metric for identifying bottlenecks.

⚠️ Note: This computation can take 2-5 minutes on large graphs.  
We normalize by graph size so values are comparable.

In [ ]:
import time

print("Computing betweenness centrality...")
print("(This may take 2-5 minutes for a graph this size)\n")

start = time.time()

# Compute betweenness centrality
# normalized=True makes values comparable across different graph sizes
# weight='weight' uses delay ratio as edge weight
betweenness = nx.betweenness_centrality(
    G,
    normalized=True,
    weight='weight'
)

elapsed = time.time() - start
print(f"✅ Betweenness centrality computed in {elapsed:.1f} seconds")
print(f"\n  Total hubs analyzed : {len(betweenness):,}")
print(f"  Max betweenness     : {max(betweenness.values()):.6f}")
print(f"  Mean betweenness    : {np.mean(list(betweenness.values())):.6f}")
print(f"  Non-zero hubs       : {sum(1 for v in betweenness.values() if v > 0):,}")

## 📊 Step 3 — Compute All Centrality Metrics

Beyond betweenness, we compute:
- **In-degree centrality** — how many routes feed into this hub
- **Out-degree centrality** — how many routes leave this hub
- **Clustering coefficient** — how tightly connected the hub's neighbors are
- **PageRank** — importance based on the importance of connected hubs

In [ ]:
print("Computing additional centrality metrics...\n")

# In and out degree centrality
in_degree_centrality  = nx.in_degree_centrality(G)
out_degree_centrality = nx.out_degree_centrality(G)

# Raw degree counts
in_degrees  = dict(G.in_degree())
out_degrees = dict(G.out_degree())
total_degrees = {n: in_degrees[n] + out_degrees[n] for n in G.nodes()}

# Clustering coefficient
print("Computing clustering coefficient...")
clustering = nx.clustering(G.to_undirected())

# PageRank
print("Computing PageRank...")
pagerank = nx.pagerank(G, weight='weight', alpha=0.85)

print("\n✅ All centrality metrics computed")
print(f"\n  Metrics computed for {len(betweenness):,} hubs:")
print(f"    - Betweenness centrality")
print(f"    - In-degree centrality")
print(f"    - Out-degree centrality")
print(f"    - Clustering coefficient")
print(f"    - PageRank")

## 🏗️ Step 4 — Build Hub Metrics DataFrame

We combine all metrics into a single dataframe  
so we can rank and compare hubs systematically.

In [ ]:
# Build hub metrics dataframe
hub_metrics = pd.DataFrame({
    'hub_id'               : list(betweenness.keys()),
    'betweenness'          : list(betweenness.values()),
    'in_degree_centrality' : [in_degree_centrality[n] for n in betweenness.keys()],
    'out_degree_centrality': [out_degree_centrality[n] for n in betweenness.keys()],
    'in_degree'            : [in_degrees[n] for n in betweenness.keys()],
    'out_degree'           : [out_degrees[n] for n in betweenness.keys()],
    'total_degree'         : [total_degrees[n] for n in betweenness.keys()],
    'clustering'           : [clustering[n] for n in betweenness.keys()],
    'pagerank'             : [pagerank[n] for n in betweenness.keys()]
})

# Add delay metrics from trip data
hub_delay = df.groupby('source_center').agg(
    avg_delay          = ('delay_ratio_clean', 'mean'),
    median_delay       = ('delay_ratio_clean', 'median'),
    total_trips        = ('trip_uuid', 'count'),
    chronic_trip_count = ('delay_ratio_clean', lambda x: (x > 1.2).sum()),
    states_served      = ('dest_state', 'nunique')
).reset_index()
hub_delay.columns = ['hub_id', 'avg_delay', 'median_delay',
                     'total_trips', 'chronic_trip_count', 'states_served']

hub_metrics = hub_metrics.merge(hub_delay, on='hub_id', how='left')
hub_metrics['chronic_rate'] = (
    hub_metrics['chronic_trip_count'] / hub_metrics['total_trips'] * 100
).fillna(0)

print(f"Hub metrics dataframe built:")
print(f"  Shape  : {hub_metrics.shape}")
print(f"  Columns: {list(hub_metrics.columns)}")
print(f"\nSample (top 5 by betweenness):")
print(hub_metrics.nlargest(5, 'betweenness')[
    ['hub_id', 'betweenness', 'total_degree', 'avg_delay', 'chronic_rate']
].to_string(index=False))

## 🎯 Step 5 — Compute Bottleneck Score

No single metric tells the full story.  
We combine multiple metrics into a **composite bottleneck score**.

**Formula:**
Bottleneck Score = (
0.40 × betweenness_normalized +
0.25 × degree_normalized +
0.20 × avg_delay_normalized +
0.15 × chronic_rate_normalized
)
**Why these weights?**
- Betweenness (40%) — structural position matters most
- Degree (25%) — connectivity drives cascading failures
- Avg delay (20%) — current delay performance
- Chronic rate (15%) — consistency of delay problem

In [ ]:
# Normalize all metrics to 0-1 range
def normalize(series):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series([0.5] * len(series), index=series.index)
    return (series - min_val) / (max_val - min_val)

hub_metrics['betweenness_norm']   = normalize(hub_metrics['betweenness'])
hub_metrics['degree_norm']        = normalize(hub_metrics['total_degree'])
hub_metrics['delay_norm']         = normalize(hub_metrics['avg_delay'].fillna(1.0))
hub_metrics['chronic_rate_norm']  = normalize(hub_metrics['chronic_rate'])

# Compute composite bottleneck score
hub_metrics['bottleneck_score'] = (
    0.40 * hub_metrics['betweenness_norm'] +
    0.25 * hub_metrics['degree_norm'] +
    0.20 * hub_metrics['delay_norm'] +
    0.15 * hub_metrics['chronic_rate_norm']
)

# Sort by bottleneck score
hub_metrics = hub_metrics.sort_values('bottleneck_score', ascending=False).reset_index(drop=True)
hub_metrics['rank'] = hub_metrics.index + 1

print("✅ Bottleneck scores computed\n")
print("Top 10 Bottleneck Hubs:\n")
print("-" * 85)
print(f"  {'Rank':<5} {'Hub ID':<18} {'Score':>8} {'Betweenness':>13} "
      f"{'Degree':>8} {'Avg Delay':>10} {'Chronic%':>10}")
print("-" * 85)

for _, row in hub_metrics.head(10).iterrows():
    print(f"  {int(row['rank']):<5} {row['hub_id']:<18} "
          f"{row['bottleneck_score']:>8.4f} "
          f"{row['betweenness']:>13.6f} "
          f"{int(row['total_degree']):>8} "
          f"{row['avg_delay']:>10.3f} "
          f"{row['chronic_rate']:>9.1f}%")
print("-" * 85)

## 🏆 Step 6 — Top 5 Bottleneck Hubs

These are the hubs that will appear in the Strategy Memo.  
For each hub we compute its estimated SLA breach contribution.

In [ ]:
top5 = hub_metrics.head(5).copy()

# Estimate SLA breach contribution
total_chronic_trips = (df['delay_ratio_clean'] > 1.2).sum()

for idx, row in top5.iterrows():
    hub_chronic = df[
        (df['source_center'] == row['hub_id']) &
        (df['delay_ratio_clean'] > 1.2)
    ].shape[0]
    top5.loc[idx, 'sla_breach_trips']       = hub_chronic
    top5.loc[idx, 'sla_breach_contribution'] = hub_chronic / total_chronic_trips * 100

print("=" * 70)
print("         TOP 5 BOTTLENECK HUBS — STRATEGY MEMO READY")
print("=" * 70)

for i, (_, row) in enumerate(top5.iterrows(), 1):
    print(f"""
  HUB #{i}: {row['hub_id']}
  {'─' * 50}
  Bottleneck Score      : {row['bottleneck_score']:.4f}
  Betweenness Centrality: {row['betweenness']:.6f}
  Total Connections     : {int(row['total_degree'])} ({int(row['in_degree'])} in, {int(row['out_degree'])} out)
  Average Delay Ratio   : {row['avg_delay']:.3f}x OSRM estimate
  Chronic Delay Rate    : {row['chronic_rate']:.1f}% of trips
  Total Trips           : {int(row['total_trips']):,}
  SLA Breach Trips      : {int(row['sla_breach_trips']):,}
  SLA Breach Contribution: {row['sla_breach_contribution']:.1f}% of all network breaches
""")

print("=" * 70)

## 📊 Step 7 — Bottleneck Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1 — Top 20 hubs by bottleneck score
top20 = hub_metrics.head(20)
colors_score = ['#E05C5C' if i < 5 else '#E8A838' if i < 10
                else '#4A90D9' for i in range(20)]

axes[0,0].barh(
    [h[:15] for h in top20['hub_id']][::-1],
    top20['bottleneck_score'].values[::-1],
    color=colors_score[::-1],
    edgecolor='white', height=0.6
)
axes[0,0].set_title('Top 20 Hubs by Bottleneck Score\n(Red = Top 5, Orange = 6-10)',
                    fontsize=11, fontweight='bold')
axes[0,0].set_xlabel('Bottleneck Score', fontsize=10)
axes[0,0].spines['top'].set_visible(False)
axes[0,0].spines['right'].set_visible(False)
axes[0,0].tick_params(axis='y', labelsize=7)

# Plot 2 — Top 20 by betweenness centrality
top20_bc = hub_metrics.nlargest(20, 'betweenness')
axes[0,1].barh(
    [h[:15] for h in top20_bc['hub_id']][::-1],
    top20_bc['betweenness'].values[::-1],
    color='#7BC8A4', edgecolor='white', height=0.6
)
axes[0,1].set_title('Top 20 Hubs by Betweenness Centrality',
                    fontsize=11, fontweight='bold')
axes[0,1].set_xlabel('Betweenness Centrality', fontsize=10)
axes[0,1].spines['top'].set_visible(False)
axes[0,1].spines['right'].set_visible(False)
axes[0,1].tick_params(axis='y', labelsize=7)

# Plot 3 — Betweenness vs Delay scatter
scatter_data = hub_metrics[hub_metrics['total_trips'] > 10].copy()
sc = axes[1,0].scatter(
    scatter_data['betweenness'],
    scatter_data['avg_delay'],
    c=scatter_data['total_degree'],
    cmap='YlOrRd',
    alpha=0.6,
    s=scatter_data['total_trips'] / scatter_data['total_trips'].max() * 200 + 20
)
plt.colorbar(sc, ax=axes[1,0], label='Total Degree')

# Highlight top 5
for _, row in top5.iterrows():
    axes[1,0].scatter(
        row['betweenness'], row['avg_delay'],
        color='#E05C5C', s=200, zorder=5,
        marker='*'
    )
    axes[1,0].annotate(
        row['hub_id'][:10],
        (row['betweenness'], row['avg_delay']),
        fontsize=7, color='#E05C5C',
        xytext=(5, 5), textcoords='offset points'
    )

axes[1,0].set_xlabel('Betweenness Centrality', fontsize=10)
axes[1,0].set_ylabel('Average Delay Ratio', fontsize=10)
axes[1,0].set_title('Betweenness vs Delay\n(Star = Top 5 Bottlenecks)',
                    fontsize=11, fontweight='bold')
axes[1,0].spines['top'].set_visible(False)
axes[1,0].spines['right'].set_visible(False)

# Plot 4 — SLA breach contribution of top 5
labels = [h[:12] for h in top5['hub_id']]
values = top5['sla_breach_contribution'].values
other  = 100 - values.sum()

pie_labels = labels + ['Rest of Network']
pie_values = list(values) + [other]
pie_colors = ['#E05C5C', '#E07B54', '#E8A838', '#4A90D9', '#7BC8A4', '#CCCCCC']

wedges, texts, autotexts = axes[1,1].pie(
    pie_values,
    labels=pie_labels,
    autopct='%1.1f%%',
    colors=pie_colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for text in texts:
    text.set_fontsize(8)
for autotext in autotexts:
    autotext.set_fontsize(8)
    autotext.set_fontweight('bold')

axes[1,1].set_title('SLA Breach Contribution\nTop 5 Bottleneck Hubs vs Rest of Network',
                    fontsize=11, fontweight='bold')

plt.suptitle('Bottleneck Hub Analysis — Delhivery Logistics Network',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/visualisations/bottleneck_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved")

## 🕸️ Step 8 — Bottleneck Network Visualization

Highlight top 5 bottleneck hubs directly on the network graph.

In [ ]:
# Get top connected nodes for visualization
top_nodes = sorted(dict(G.degree()).items(),
                   key=lambda x: x[1], reverse=True)[:80]
top_node_ids = [n for n, d in top_nodes]
subG = G.subgraph(top_node_ids)

pos = nx.spring_layout(subG, k=2, seed=42)

top5_ids = set(top5['hub_id'].values)

# Node sizes and colors
node_sizes  = []
node_colors = []
for n in subG.nodes():
    degree = dict(subG.degree())[n]
    if n in top5_ids:
        node_sizes.append(degree * 150 + 500)
        node_colors.append('#E05C5C')
    else:
        score = hub_metrics[hub_metrics['hub_id'] == n]['bottleneck_score'].values
        node_sizes.append(degree * 50 + 100)
        node_colors.append('#4A90D9' if len(score) == 0 else
                          '#E8A838' if score[0] > 0.3 else '#7BC8A4')

# Edge colors
edge_colors = []
for u, v, d in subG.edges(data=True):
    delay = float(d.get('median_delay', 1.0))
    if delay > 1.5:
        edge_colors.append('#E05C5C')
    elif delay > 1.2:
        edge_colors.append('#E8A838')
    else:
        edge_colors.append('#AAAAAA')

fig, ax = plt.subplots(figsize=(18, 14))

nx.draw_networkx_edges(
    subG, pos, ax=ax,
    edge_color=edge_colors,
    width=0.8, alpha=0.5,
    arrows=True, arrowsize=6,
    connectionstyle='arc3,rad=0.1'
)

nx.draw_networkx_nodes(
    subG, pos, ax=ax,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.9
)

# Labels only for top 5 bottleneck hubs
labels_top5 = {n: f"★ {n[:10]}" for n in subG.nodes() if n in top5_ids}
nx.draw_networkx_labels(
    subG, pos, labels_top5, ax=ax,
    font_size=8, font_color='#8B0000',
    font_weight='bold'
)

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [
    Patch(facecolor='#E05C5C', label='Top 5 Bottleneck Hubs ★'),
    Patch(facecolor='#E8A838', label='High Risk Hubs'),
    Patch(facecolor='#7BC8A4', label='Normal Hubs'),
    Line2D([0], [0], color='#E05C5C', linewidth=2, label='Chronic corridor (>1.5x)'),
    Line2D([0], [0], color='#E8A838', linewidth=2, label='Delayed corridor (1.2-1.5x)'),
    Line2D([0], [0], color='#AAAAAA', linewidth=1, label='Normal corridor (<1.2x)')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10,
          framealpha=0.9)

ax.set_title(
    'Delhivery Logistics Network — Bottleneck Hubs Highlighted\n'
    '★ = Top 5 Structural Bottlenecks',
    fontsize=14, fontweight='bold', pad=20
)
ax.axis('off')

plt.tight_layout()
plt.savefig('../outputs/visualisations/bottleneck_network.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Bottleneck network graph saved")

## 🔍 Step 9 — Corridor Audit for Top 5 Hubs

For each bottleneck hub, we identify its worst corridors.  
These become the specific intervention targets in the strategy memo.

In [ ]:
print("=" * 70)
print("    CORRIDOR AUDIT — TOP 5 BOTTLENECK HUBS")
print("=" * 70)

for i, (_, row) in enumerate(top5.iterrows(), 1):
    hub = row['hub_id']

    # Outbound corridors from this hub
    outbound = df[df['source_center'] == hub].groupby(
        ['source_center', 'destination_center']
    ).agg(
        trips        = ('delay_ratio_clean', 'count'),
        median_delay = ('delay_ratio_clean', 'median'),
        chronic_rate = ('delay_ratio_clean', lambda x: (x > 1.2).mean() * 100)
    ).reset_index().sort_values('median_delay', ascending=False)

    print(f"\n  HUB #{i}: {hub}")
    print(f"  Worst outbound corridors:")
    print(f"  {'Destination':<20} {'Trips':>6} {'Med Delay':>10} {'Chronic%':>10}")
    print(f"  {'-'*50}")

    for _, corr in outbound.head(5).iterrows():
        print(f"  {corr['destination_center'][:20]:<20} "
              f"{int(corr['trips']):>6} "
              f"{corr['median_delay']:>10.3f} "
              f"{corr['chronic_rate']:>9.1f}%")

print("\n" + "=" * 70)

In [ ]:
# Save hub metrics to CSV
hub_metrics.to_csv('../outputs/model_results/hub_bottleneck_scores.csv', index=False)
top5.to_csv('../outputs/model_results/top5_bottleneck_hubs.csv', index=False)

print("✅ Files saved:")
print("   hub_bottleneck_scores.csv — all hub rankings")
print("   top5_bottleneck_hubs.csv  — top 5 for strategy memo")

In [ ]:
print("=" * 70)
print("         BOTTLENECK ANALYSIS COMPLETE")
print("=" * 70)
print(f"""
NETWORK OVERVIEW
  Total hubs analyzed      : {len(hub_metrics):,}
  Total corridors          : {G.number_of_edges():,}

TOP 5 BOTTLENECK HUBS
""")

for i, (_, row) in enumerate(top5.iterrows(), 1):
    print(f"  #{i} {row['hub_id']:<20} "
          f"Score: {row['bottleneck_score']:.4f} | "
          f"SLA Contribution: {row['sla_breach_contribution']:.1f}%")

total_contribution = top5['sla_breach_contribution'].sum()
print(f"""
  Combined SLA breach contribution: {total_contribution:.1f}%
  → Fixing these 5 hubs addresses {total_contribution:.1f}% of all network delays

FILES SAVED
  hub_bottleneck_scores.csv
  top5_bottleneck_hubs.csv
  bottleneck_analysis.png
  bottleneck_network.png
""")
print("=" * 70)
print("  NEXT STEP → 05_baseline_model.ipynb")
print("=" * 70)

---
## ✅ Bottleneck Analysis Complete

### Key findings:
- Top 5 bottleneck hubs identified with composite scores
- SLA breach contribution quantified for each hub
- Corridor-level audit completed for targeted interventions
- All findings ready for Strategy Memo

### Files saved:
- `outputs/model_results/hub_bottleneck_scores.csv`
- `outputs/model_results/top5_bottleneck_hubs.csv`

---
### ➡️ Next: `05_baseline_model.ipynb`